In [42]:
import pandas as pd
import numpy as np
n_items = 10
n_users = 10
k = 10 
ten = [str(j).zfill(2) for j in list(range(1,k+1))]
ten

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']

In [43]:
labs = ['test'+i for i in ten]
labs.insert(0,'orig')
url = "https://raw.githubusercontent.com/park61/imputation/main/data/"
url_dict = {}
url_dict["orig"] = url+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
for i in ten:
    url_dict["test{0}".format(i)] = url+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'

In [44]:
print(labs)
for lab in labs:
  print(url_dict[lab])

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_original_matrix.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data01.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data02.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data03.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data04.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data05.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data06.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data07.csv
https://raw.githubusercontent.com/park61/imputation/main/data/10-by-10_10_fold_test_data08.csv
https://raw.githubusercontent.com/park61

In [45]:
df_dict = {}
for lab in labs:
  df = pd.read_csv(url_dict[lab])
  df_dict[lab] = df.set_index('item_id')

In [46]:
# compute sparcity of the original data
#df_dict["orig"].head()
#pivot_df = df_dict["orig"].pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
n_row = df_dict["orig"].shape[0]
n_col = df_dict["orig"].shape[1]
Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
sparsity = 1 - num_Obs / (n_row * n_col)
print(sparsity)

0.10999999999999999


In [47]:
import numpy as np
col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
#print(col_max)

In [48]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out#, maxs, probs

In [49]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [50]:
# A subroutine that calculates $\tau_b$ coefficient for columns $i$ and $j$ of the input dataframe `df`.

def tau_b(df, i, j):
  import scipy.stats as stats
  dat = df.copy()
  keep_ind = []
  for k in range(len(df)):
    if df.isna().iloc[k,i]+df.isna().iloc[k,j] == 0:
      keep_ind.append(k)
  dat_out = dat.iloc[keep_ind,:]
  tau, p = stats.kendalltau(dat_out.iloc[:,i],dat_out.iloc[:,j])
  return tau

In [51]:
def tau_b_mat(df):
  import numpy as np
  n = df.shape[1]
  tau_mat = np.empty(shape=(n, n), dtype='double')
  for i in range(n):
    for j in range(n):
      tau_mat[i,j] = tau_b(df, i, j)
  return tau_mat

# Subroutine: Column-wise aggregation 
It takes a normalized dataframe `df` and the column label `i` and returns the aggregated column by weighted average:
$$
s_i = \frac{\sum_l w_l X_i}{\sum_l w_l}
$$`

In [52]:
def c_agg(df, w):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]

  agg_mat = np.empty(shape=(m, n), dtype='double')
  masked_df = np.ma.masked_array(df, np.isnan(df))

  for i in range(m):
    for j in range(n):
      agg_mat[i,j] = np.ma.average(masked_df[i,:], weights=w[:,j])
  return agg_mat

Subroutine: Take a column of a missing data and return the imputed one based on monotonization

In [53]:
def monotonization(input):

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB
  import numpy as np

  n = len(input)
  c = np.nanmax(input)
  k = n - np.isnan(input).sum()

  ind = np.where(1-np.isnan(input))
  K = ind[0]

  #m = gp.Model('Monotonization')
  x = m.addVars(n, lb=1, ub=c, name="x")
  u = m.addVars(n, lb=0, name="u")
  w = m.addVars(n, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in K), GRB.MINIMIZE)

  m.addConstr(1 <= x[0])
  m.addConstr(x[n-1] <= c)
  m.addConstrs(x[i]<=x[i+1] for i in range(n-1))
  m.addConstrs(x[j]-input[j] == u[j]-w[j] for j in K)

  m.optimize()

  x_out = m.getAttr('X', x)

  for i in K:
    x_out[i] = input[i]

  return x_out

In [54]:
def rmse_cal(df, corr_mat, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)
  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()
  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [32]:
def rmse_cal2(df, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)

  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()

  w = np.array(df.corr())
  w[w<0]=0
  for i in range(len(w)):
    w[i,i]=0

  corr_mat = w

  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [33]:
def nested_quantile(df, a_min, a_max, n_level, RMSE_max):
  import numpy as np
  Q = np.zeros(5)
  Q[0] = a_min
  Q[4] = a_max
  beta = abs(Q[0]-Q[4])/4
  for j in range(1,4):
    Q[j] = Q[0] + j*beta

  # RMSE_L = rmse_calculator(df,Q[0])
  # RMSE_R = rmse_calculator(df,Q[4])
  RMSE_L = rmse_cal2(df,Q[0])
  RMSE_R = rmse_cal2(df,Q[4])

  # When RMSE_M=1, it is temporary
  RMSE_M = 1

  for i in range(n_level):

    RMSE = pd.DataFrame([[Q[0],RMSE_L],[Q[1],None],[Q[2],RMSE_M],[Q[3],None],[Q[4],RMSE_R]])

    for k in range(1,4):
      if k != 2 or RMSE_M == 1:
        # RMSE.iloc[k,1] =  rmse_calculator(df,Q[k])
        RMSE.iloc[k,1] =  rmse_cal2(df,Q[k])
      else:
        RMSE.iloc[k,1] = RMSE_M
    
    RMSE_sorted = RMSE.sort_values(by=[1])
        
    #print('RMSE Sorted:')
    #display(RMSE_sorted)
    
    alpha = RMSE_sorted.iloc[0,0]

    if RMSE_sorted.iloc[0,0] == Q[0]:
      Q[4] = Q[1]
      RMSE_R = RMSE.iloc[1,1] 
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[4]:
      Q[0] = Q[3] 
      RMSE_L = RMSE.iloc[3,1]
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[1]:
      Q[4] = Q[2]
      RMSE_M = RMSE.iloc[1,1]
      RMSE_R = RMSE.iloc[2,1]
    elif RMSE_sorted.iloc[0,0] == Q[2]:
      Q[0] = Q[1]
      Q[4] = Q[3]
      RMSE_L = RMSE.iloc[1,1]
      RMSE_M = RMSE.iloc[2,1]
      RMSE_R = RMSE.iloc[3,1]
    elif RMSE_sorted.iloc[0,0] == Q[3]:
      Q[0] = Q[2]
      RMSE_L = RMSE.iloc[2,1]
      RMSE_M = RMSE.iloc[3,1]

    beta = abs(Q[0]-Q[4])/4
    for j in range(1,4):
      Q[j] = Q[0] + j*beta

  return alpha

In [55]:
def power_finder(df, corr_mat, step_size, init_power, max_power):
  import math
  current_power = init_power
  opt_power = current_power
  opt_rmse = rmse_cal(df, corr_mat, opt_power)
  n_iter = math.floor((max_power-init_power)/step_size)
  for i in range(n_iter):
    current_power += step_size
    new_rmse = rmse_cal(df, corr_mat, current_power)
    if opt_rmse > new_rmse:
      opt_power = current_power
      opt_rmse = new_rmse
  return opt_power

In [56]:
df_orig = df_dict["orig"]
mm = df_orig.shape[0]
nn = df_orig.shape[1]

corr_mat = df_orig.corr()
import numpy as np
temp_corr = np.copy(corr_mat)

In [57]:
#@title Run MonoImoute algorithm
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

acc_list = []
rmse_list = []
mad_list = []
time_list = []

powers = [] #for monimpute

df_orig = df_dict["orig"]
df_orig.columns = range(df_orig.shape[1])

corr_mat = df_orig.corr()
temp_corr = np.copy(corr_mat)
temp_corr[temp_corr < 0] = 0
#print(np.round(temp_corr,2))
#print(np.round(corr_mat,2))
np.fill_diagonal(temp_corr, 0)
orig_corr = np.copy(temp_corr)


mm = df_orig.shape[0]
nn = df_orig.shape[1]

labs_test = labs[1:]

a_min = 0
a_max = 16
n_level = 4
RMSE_max = 1

count = 0
for lab in labs_test:

  print(lab)
  count += 1
  print(count)
  orig_corr = np.copy(temp_corr)
  df = df_dict[lab]
  import numpy as np
  masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
  #df, maxs = normal1(masked_df)
  #df, maxs = normal2(masked_df)
  #df, maxs, probs = normal3(masked_df)
  df_norm = normal3(masked_df)


  # from missingpy import MissForest
  # imputer = MissForest()

  # from fancyimpute import SoftImpute
  # imputer = SoftImpute()

  start = time_lib.time()
  #df_sampled = df_norm.sample(n=10)
  #df_norm = df
  #power = power_finder(df_sampled, orig_corr, 1, 1, 20)
  power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
  powers.append(power)
  #power = 1

  #corr = np.power(orig_corr, power)  #issue: With irrational exponents 𝑟 and negative real numbers 𝑥 (2023.03.27)
  corr = np.power(orig_corr, power)  
  agg_mat = c_agg(df_norm, corr)

  df_agg_mat = pd.DataFrame(agg_mat)
  df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
  df_agg_mat.index = df.index
  df_agg = pd.concat([df, df_agg_mat], axis=1)
  #print(df_agg.shape)
  id_column = pd.DataFrame(range(mm))
  id_column.index = df_agg.index
  df_agg = pd.concat([id_column, df_agg], axis=1)
  #print(df_agg.shape)
  df_agg.columns = range(2*nn+1)
  
  

  for i in range(nn):
    df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
    index_i = df_agg_sorted.index
    vec = np.array(df_agg_sorted.iloc[:,i+1])
    out = monotonization(vec)
    out_list = []
    for i in range(len(out)):
      out_list.append(out[i])

    out_df = pd.DataFrame(out_list, index=index_i)
    df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

  df_agg.columns = range(3*nn+1)
  df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
  imputed = df_agg_sorted_final.iloc[:,2*nn+1:]
  
  display(imputed)
  #save the result data
 
  if count<10:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
  else:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
    
  imputed.to_csv(filename)
    
  imputed.columns = range(nn)

  end = time_lib.time()
  time = end - start

  df_orig.index = range(mm)
  imputed.index = range(mm)

  df_orig.columns = range(nn)
  imputed.columns = range(nn)

  diff = df_orig - imputed
  sq_diff = diff ** 2
  abs_diff = abs(diff)

  n_match = 0
  for i in range(mm):
    for j in range(nn):
      if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
        n_match += 1
  acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(acc)  
  rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
  mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(rmse)
  print(mad)
  print(time)
  acc_list.append(acc)
  rmse_list.append(rmse)
  mad_list.append(mad)
  time_list.append(time)

test01
1


,21,22,23,24,25,26,27,28,29,30
item_id,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0
258,5.0,2.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0
100,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,4.0,1.0
181,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,4.0
294,5.0,3.0,2.0,4.0,4.0,4.0,1.0,4.0,3.0,4.0
286,5.0,3.0,3.0,4.0,5.0,5.0,3.0,5.0,3.0,4.0
288,5.0,3.0,1.0,3.0,4.0,5.0,2.0,4.0,3.0,3.0
1,5.0,2.0,4.0,4.0,5.0,5.0,2.0,5.0,3.0,3.0
300,5.0,3.0,1.0,4.0,4.0,4.0,1.0,1.0,3.0,4.0


0.75
0.7905694150420949
0.375
0.7887978553771973
test02
2


,21,22,23,24,25,26,27,28,29,30
item_id,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,4.0
258,5.0,2.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0
100,5.0,3.0,5.0,4.0,5.0,5.0,4.0,5.0,4.0,1.0
181,5.0,3.0,5.0,5.0,5.0,5.0,2.0,5.0,3.0,4.0
294,5.0,3.0,2.0,1.0,4.0,4.0,1.0,4.0,3.0,4.0
286,5.0,3.0,3.0,4.0,5.0,5.0,3.0,5.0,3.0,4.0
288,5.0,4.0,1.0,3.0,4.0,5.0,2.0,4.0,3.0,3.0
1,5.0,2.0,3.0,4.0,5.0,5.0,2.0,5.0,3.0,3.0
300,5.0,2.0,1.0,1.0,4.0,4.0,1.0,1.0,3.0,1.0


0.3333333333333333
1.5634719199411433
1.1111111111111112
0.29572319984436035
test03
3


...

0.5555555555555556
0.8819171036881969
0.5555555555555556
0.314319372177124
test09
9


,21,22,23,24,25,26,27,28,29,30
item_id,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,4.0
258,5.0,1.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0
100,5.0,3.0,5.0,4.0,5.0,5.0,4.0,5.0,4.0,1.0
181,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,4.0
294,5.0,3.0,2.0,4.0,4.0,4.0,1.0,4.0,3.0,4.0
286,5.0,3.0,3.0,4.0,5.0,5.0,3.0,5.0,3.0,4.0
288,5.0,3.0,1.0,3.0,4.0,5.0,2.0,4.0,3.0,3.0
1,5.0,2.0,3.0,4.0,5.0,5.0,2.0,5.0,3.0,3.0
300,5.0,3.0,1.0,4.0,4.0,4.0,1.0,1.0,3.0,4.0


0.6666666666666666
0.816496580927726
0.4444444444444444
0.30043458938598633
test10
10


,21,22,23,24,25,26,27,28,29,30
item_id,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0
258,5.0,1.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0
100,5.0,3.0,3.0,4.0,5.0,5.0,4.0,5.0,4.0,1.0
181,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,4.0
294,5.0,3.0,2.0,4.0,4.0,5.0,1.0,4.0,3.0,4.0
286,5.0,3.0,3.0,4.0,5.0,5.0,3.0,5.0,3.0,4.0
288,5.0,3.0,1.0,3.0,4.0,5.0,2.0,4.0,3.0,3.0
1,5.0,2.0,3.0,4.0,5.0,5.0,2.0,5.0,1.0,1.0
300,5.0,3.0,1.0,4.0,4.0,4.0,1.0,1.0,3.0,4.0


0.4444444444444444
1.247219128924647
0.8888888888888888
0.31871628761291504


In [59]:
diff

,0,1,2,3,4,5,6,7,8,9
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0
5,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0
8,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
9,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0


In [66]:
sq_diff = diff ** 2
sq_diff

,0,1,2,3,4,5,6,7,8,9
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0
8,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
9,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0


In [68]:
sq_diff.sum().sum()

14.0

In [69]:
(df.isna().sum().sum()-df_orig.isna().sum().sum())

9

In [71]:
(sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) 

1.5555555555555556

In [73]:
np.sqrt((sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) )

1.247219128924647

In [70]:
(sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5

1.247219128924647

In [61]:
df

,405,655,13,450,276,416,537,303,234,393
item_id,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,5.0,4.0,NaN,4.0,5.0
258,NaN,NaN,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0
100,NaN,3.0,NaN,4.0,5.0,5.0,4.0,5.0,4.0,1.0
181,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,4.0
294,NaN,3.0,2.0,4.0,4.0,NaN,1.0,4.0,3.0,4.0
286,NaN,NaN,3.0,4.0,NaN,NaN,3.0,5.0,3.0,NaN
288,5.0,3.0,1.0,3.0,4.0,5.0,2.0,4.0,3.0,3.0
1,NaN,2.0,3.0,4.0,5.0,5.0,2.0,5.0,NaN,NaN
300,NaN,3.0,1.0,4.0,4.0,4.0,1.0,NaN,3.0,NaN


In [63]:
df.isna().sum().sum()

20

In [62]:
df_orig

,0,1,2,3,4,5,6,7,8,9
0,5.0,4.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0
1,NaN,2.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0
2,NaN,3.0,5.0,4.0,5.0,5.0,4.0,5.0,4.0,1.0
3,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,4.0
4,NaN,3.0,2.0,4.0,4.0,4.0,1.0,4.0,3.0,4.0
5,NaN,3.0,3.0,4.0,NaN,5.0,3.0,5.0,3.0,NaN
6,5.0,3.0,1.0,3.0,4.0,5.0,2.0,4.0,3.0,3.0
7,NaN,2.0,3.0,4.0,5.0,5.0,2.0,5.0,3.0,3.0
8,NaN,3.0,1.0,4.0,4.0,4.0,1.0,1.0,3.0,NaN
9,NaN,3.0,5.0,3.0,4.0,5.0,1.0,3.0,NaN,4.0


In [65]:
df_orig.isna().sum().sum()

11

In [37]:
diff = df_orig - imputed
  sq_diff = diff ** 2
  abs_diff = abs(diff)

  n_match = 0
  for i in range(mm):
    for j in range(nn):
      if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
        n_match += 1
  acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(acc)  
  rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
  mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())

10

In [60]:
n_match

4

In [38]:
# print(sparsity)

print("Accuracy")
for i in acc_list:
  print(i)
print("\nRMSE")
for i in rmse_list:
  print(i)
print("\nMAD")
for i in mad_list:
  print(i)
print("\nTime")
for i in time_list:
  print(i)

Accuracy
0.5238900634249471
0.505708245243129
0.5378435517970401
0.5188160676532769
0.5116279069767442
0.5331923890063425
0.5399577167019027
0.5226215644820296
0.5114116652578191
0.5333896872358411

RMSE
0.9585489385802255
0.9838011452758575
0.9492402659245084
0.9738656870027577
1.0056920455518468
0.9214383461701666
0.9341987329938275
0.9570037809676978
0.9853105375729218
0.9452465944188083

MAD
0.6
0.626215644820296
0.5830866807610994
0.6109936575052854
0.6300211416490487
0.5758985200845665
0.5784355179704017
0.5995771670190275
0.6234150464919695
0.5841081994928149

Time
51.60154867172241
50.53220510482788
50.649329662323
53.29932737350464
117.47656059265137
86.48427891731262
51.097710371017456
51.7206757068634
52.717774391174316
50.44260263442993


In [39]:
df = pd.DataFrame(data=[acc_list, rmse_list,mad_list,time_list])
df.T

,0,1,2,3
0,0.523890,0.958549,0.600000,51.601549
1,0.505708,0.983801,0.626216,50.532205
2,0.537844,0.949240,0.583087,50.649330
3,0.518816,0.973866,0.610994,53.299327
4,0.511628,1.005692,0.630021,117.476561
5,0.533192,0.921438,0.575899,86.484279
6,0.539958,0.934199,0.578436,51.097710
7,0.522622,0.957004,0.599577,51.720676
8,0.511412,0.985311,0.623415,52.717774
9,0.533390,0.945247,0.584108,50.442603


In [40]:
print(powers)

[6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.0, 6.5, 6.5, 6.5]
